In [37]:
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
import country_converter as coco

In [89]:
df = pd.read_parquet("../data/final_clean.csv")

In [90]:
df

,conflict_id,year,side_a,side_a_2nd,side_b,side_b_2nd,iso3_a,iso3_a_2nd,iso3_b,iso3_b_2nd,...,version_prio,prio_merge,bd_best,bd_low,bd_high,deaths_best,deaths_low,deaths_high,deaths_estimate,deaths_source
0,200,1946,Government of Bolivia,None,Popular Revolutionary Movement,None,BOL,None,None,None,...,2009-4,both,NaN,NaN,NaN,1000.0,1000.0,9999.0,1000.0,PRIO_v3.1 (point)
1,200,1946,Government of Bolivia,None,Popular Revolutionary Movement,None,BOL,None,None,None,...,2009-4,both,NaN,NaN,NaN,1000.0,1000.0,9999.0,1000.0,PRIO_v3.1 (point)
2,200,1946,Government of Bolivia,None,Popular Revolutionary Movement,None,BOL,None,None,None,...,2009-4,both,NaN,NaN,NaN,1000.0,1000.0,9999.0,1000.0,PRIO_v3.1 (point)
3,200,1946,Government of Bolivia,None,Popular Revolutionary Movement,None,BOL,None,None,None,...,2009-4,both,NaN,NaN,NaN,1000.0,1000.0,9999.0,1000.0,PRIO_v3.1 (point)
4,200,1946,Government of Bolivia,None,Popular Revolutionary Movement,None,BOL,None,None,None,...,2009-4,both,NaN,NaN,NaN,1000.0,1000.0,9999.0,1000.0,PRIO_v3.1 (point)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438420,16470,2025,Government of Israel,None,Government of Yemen (North Yemen),None,ISR,None,YEM,None,...,None,left_only,164.0,94.0,197.0,164.0,94.0,197.0,164.0,UCDP_v26.1
438421,16470,2025,Government of Israel,None,Government of Yemen (North Yemen),None,ISR,None,YEM,None,...,None,left_only,164.0,94.0,197.0,164.0,94.0,197.0,164.0,UCDP_v26.1
438422,16470,2025,Government of Israel,None,Government of Yemen (North Yemen),None,ISR,None,YEM,None,...,None,left_only,164.0,94.0,197.0,164.0,94.0,197.0,164.0,UCDP_v26.1
438423,16470,2025,Government of Israel,None,Government of Yemen (North Yemen),None,ISR,None,YEM,None,...,None,left_only,164.0,94.0,197.0,164.0,94.0,197.0,164.0,UCDP_v26.1


In [92]:
df[['country_code', 'year', 'conflict_id', 'has_conflict_mention','side_a', 'side_a_2nd', 'side_b', 'side_b_2nd',
       'iso3_a', 'iso3_a_2nd', 'iso3_b', 'iso3_b_2nd', 'iso3_loc',
       'intensity_level', 'cumulative_intensity', 'type_of_conflict',
       'main_actor', 'secondary_actor',
       'conflict_host', 'side_a_ideals_diff', 'side_b_ideals_diff',
       'side_a_dist', 'side_b_dist', 'location', 'incompatibility',
       'territory_name', 'start_date', 'start_prec', 'start_date2',
       'start_prec2', 'ep_end', 'ep_end_date', 'ep_end_prec', 'region',
       'version', 'bdeadlow', 'bdeadhig', 'bdeadbes', 'annualdata', 'source',
       'bdversion', 'location_prio', 'incomp', 'terr', 'int', 'cumint', 'type',
       'startdate', 'startprec', 'startdate2', 'startprec2', 'epend',
       'ependdate', 'ependprec', 'region_prio', 'version_prio', 'prio_merge',
       'bd_best', 'bd_low', 'bd_high', 'deaths_best', 'deaths_low',
       'deaths_high', 'deaths_estimate', 'deaths_source']].sort_values(['country_code', 'year', 'conflict_id'])

,country_code,year,conflict_id,has_conflict_mention,side_a,side_a_2nd,side_b,side_b_2nd,iso3_a,iso3_a_2nd,...,version_prio,prio_merge,bd_best,bd_low,bd_high,deaths_best,deaths_low,deaths_high,deaths_estimate,deaths_source
74,AFG,1952,200,False,Government of Bolivia,None,MNR,None,BOL,None,...,2009-4,both,NaN,NaN,NaN,NaN,450.0,3000.0,1725.0,PRIO_v3.1 (range)
474,AFG,1952,201,False,Government of France,None,Khmer Issarak,None,FRA,None,...,2009-4,both,NaN,NaN,NaN,NaN,25.0,999.0,512.0,PRIO_v3.1 (range)
4088,AFG,1952,208,False,Government of France,None,Lao Issara,None,FRA,None,...,2009-4,both,NaN,NaN,NaN,NaN,25.0,999.0,512.0,PRIO_v3.1 (range)
4422,AFG,1952,209,False,Government of Philippines,None,Huk,None,PHL,None,...,2009-4,both,NaN,NaN,NaN,1077.0,1000.0,2154.0,1077.0,PRIO_v3.1 (point)
14540,AFG,1952,216,False,Government of France,None,Viet minh,None,FRA,None,...,2009-4,both,NaN,NaN,NaN,41947.0,23566.0,66667.0,41947.0,PRIO_v3.1 (point)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
435735,ZWE,2025,15256,False,Government of Benin,None,JNIM,None,BEN,None,...,None,left_only,149.0,140.0,193.0,149.0,140.0,193.0,149.0,UCDP_v26.1
436887,ZWE,2025,16038,False,Government of Mali,Government of Russia (Soviet Union),FLA,None,MLI,RUS,...,None,left_only,82.0,81.0,144.0,82.0,81.0,144.0,82.0,UCDP_v26.1
437461,ZWE,2025,16069,False,Government of Ethiopia,None,Fano,None,ETH,None,...,None,left_only,3835.0,2275.0,12282.0,3835.0,2275.0,12282.0,3835.0,UCDP_v26.1
437843,ZWE,2025,16099,False,"Government of United Kingdom, Government of Un...","Government of Australia, Government of Bahrain...",Government of Yemen (North Yemen),None,"GBR, USA","CAN, NLD, NOR, BHR, LKA, SGP, AUS, NZL",...,None,left_only,781.0,337.0,1044.0,781.0,337.0,1044.0,781.0,UCDP_v26.1


In [96]:
df[['country_code', 'year', 'conflict_id', 'iso3_loc', 'has_conflict_mention', 'intensity_level', 'side_a_ideals_diff',
       'side_a_dist', 'deaths_best', 'deaths_low',
       'deaths_high']].sort_values(['year', 'country_code', 'conflict_id']).head(55)

,country_code,year,conflict_id,iso3_loc,has_conflict_mention,intensity_level,side_a_ideals_diff,side_a_dist,deaths_best,deaths_low,deaths_high
0,ARG,1946,200,BOL,True,2,0.295676,1865.8390,1000.0,1000.0,9999.0
227,ARG,1946,201,KHM,False,1,0.425127,10932.3400,NaN,25.0,999.0
561,ARG,1946,202,CHN,False,2,NaN,19110.1300,200000.0,16666.0,333333.0
713,ARG,1946,203,GRC,False,2,1.055082,11772.0400,38500.0,6000.0,40000.0
865,ARG,1946,204,IDN,False,1,1.034770,11434.1000,NaN,655.0,5000.0
1017,ARG,1946,205,IRN,False,1,0.532516,13881.9600,25.0,0.0,1000.0
3763,ARG,1946,206,IRN,False,1,0.532516,13881.9600,20.0,20.0,1000.0
3802,ARG,1946,207,ISR,False,1,1.457434,11136.9600,92.0,92.0,373.0
3841,ARG,1946,208,LAO,False,1,0.425127,10932.3400,NaN,25.0,999.0
4175,ARG,1946,209,PHL,False,2,0.322147,17783.1100,1077.0,1000.0,2154.0


In [104]:
df[['country_code', 'year', 'conflict_id', 'iso3_loc', 'has_conflict_mention', 'intensity_level', 'side_a_ideals_diff',
       'side_a_dist', 'deaths_best', 'deaths_low',
       'deaths_high']].groupby(['country_code', 'year']).agg(sum).head(55)

C:\Users\SemKj\AppData\Local\Temp\ipykernel_11704\3872609430.py:3: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  'deaths_high']].groupby(['country_code', 'year']).agg(sum).head(55)


conflict_id  \
country_code year                
AFG          1952         3102   
             1957         4122   
             1958         4382   
             1959         3897   
             1960         3804   
             1961         5341   
             1962         5076   
             1963         5061   
             1964         6560   
             1965         7116   
             1966         7433   
             1968         6907   
             1969        18848   
             1971         7941   
             1972         7160   
             1973        18316   
             1974         7913   
             1975         8759   
             1976         9336   
             1977        10766   
             1978        11602   
             1979        12442   
             1980        12386   
             1981        12632   
             1982        13059   
             1983        13037   
             1984        12500   
             1985        11462   
             1986        13477   
             1987        14308   
             1988        11870   
             1989        12641   
             1990        15512   
             1991        39437   
             1992        16808   
             1993        14897   
             1994        16693   
             1995        25005   
             1996        13342   
             1997        13018   
             1998        13673   
             1999        13673   
             2000        24263   
             2001        36963   
             2002        10931   
             2003        11048   
             2004        11350   
             2005        32799   
             2006        32913   
             2007        22911   
             2008        24164   
             2009        23417   
             2010        10388   
             2011        45138   
             2012        67667   

                                                            iso3_loc  \
country_code year                                                      
AFG          1952  BOLKHMLAOPHLVNMMMRMMRMMRMMRMYSISRPRK, KORGBR, ...   
             1957  MMRMMRMMRMMRMYSISRCUBDZACYPVNMINDMMRCMRHND, NI...   
             1958  MMRMMRMMRMMRCHN, TWNISRCUBIDNDZACYPVNMINDCMRMR...   
             1959   MMRMMRMMRMMRISRCHNIDNDZACYPVNMINDCMRIRQMYSLAOMMR   
             1960      MMRMMRMMRMMRISRIDNDZAVNMMYSLAOMMRCODETHNPLCMR   
             1961  MMRMMRMMRMMRMMRISRCUBIDNDZAVNMINDLAOAGOMMRCODC...   
             1962  MMRMMRMMRYEMMMRISRDZAVNMINDAGOMMRCODCODNPLFRAI...   
             1963  MMRMMRMMRYEMMMRGTMISRARGVNMINDIRQLAOAGOMMRIRQM...   
             1964  IND, PAKMMRMMRMMRYEMMMRISRVNMINDLAOAGOMMRSOM, ...   
             1965  IND, PAKMMRMMRMMRYEMMMRGTMISRINDLAOAGOMMRIRQET...   
             1966  IRNMMRMMRMMRYEMMMRGTMISRINDLAOAGOMMRIRQETHGNBM...   
             1968  IRNMMRMMRMMRYEMMMRGTMISRINDLAOAGOMMRIRQETHGNBS...   
             1969  PHLMMRMMRMMRINDYEMMMRGTMISRLAOAGOMMRIRQETHGNBS...   
             1971  PHLIND, PAKMMRMMRMMRINDMMRGTMISRLAOAGOETHGNBSD...   
             1972  PHLMMRMMRMMRMMRGTMISRLAOAGOMMRETHGNBSDNMOZTCDC...   
             1973  PHLMMRMMRMMRMMRGTMISRLAOAGOMMRIRQETHGNBMOZCOLV...   
             1974  PHLMMRMMRMMRMMRGTMISRTHAARGMYSAGOIRQETHMOZCOLV...   
             1975  PHLMMRMMRMMRMMRGTMISRTHAARGLBNMYSIRQETHCOLVNM,...   
             1976  PHLMMRMMRMMRMMRGTMISRTHAARGLBNMMRETHIRQETHTCDC...   
             1977  PHLMMRMMRMMRMMRGTMISRTHAARGMMRETHSOM, ETHIRQET...   
             1978  PHLMMRMMRMMRMMRGTMISRTHAMMRETHSOM, ETHIRQETHCO...   
             1979  IRNPHLMMRMMRYEMMMRGTMISRTHAMMRETHIRQETHTCDCOLZ...   
             1980  IRNPHLMMRMMRYEMMMRGTMISRTHAMMRETHSOM, ETHIRQET...   
             1981  IRNPHLMMRMMRYEMMMRGTMISRTHAMYSMMRETHIRQETHTCDC...   
             1982  IRNPHLMMRMMRYEMMMRGTMISRTHAIRQLBNMMRETHIRQETHV...   
             1983  IRNPHLMMRMMRMMRGTMISRIRQLBNMMRETHIRQETHTCDCOLP...   
             1984  IRNPHLIND, PAKMMRMMRMMRGTMISRIRQLBNMMRETHIRQE